In [ ]:
import sys

sys.path.append("../")
sys.path.append("../src/")


In [ ]:
import json

import torch
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from attrdict import AttrDict
from sklearn.manifold import TSNE
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader
from matplotlib.colors import CSS4_COLORS

from src.models import *
from src.datasets import *


In [ ]:
# config
model_weight = "../result/all_data/tmp/model_epoch39.pth"
model_name = "efficientnet-b5"
batch_size = 64
thread = 10
gpu = 0


In [ ]:
# modelの読み込み
device = torch.device(gpu)
model = get_model(model_name, 122, 2, 11)
model = model.to(device)
model.load_state_dict(torch.load(model_weight))
model.eval()
pass


In [ ]:
# datasetの読み込み
database = AttrDict()
database.metadata = json.load(open("../data/train_meta.json"))
database.image_size = model2size[model_name]
database.mean, database.std = mean_std(model_name)
dataset = glob.glob("../data/train/*/*.jpg")
dataset = Dataset(database, dataset, train=False)
dataloader = DataLoader(
    dataset,
    shuffle=True,
    batch_size=batch_size,
    num_workers=thread,
    pin_memory=True,
)
label_encoder = dataset.model_number_encoder


In [ ]:
# 埋め込み
embedding_info = np.zeros((len(dataset), model.embedding_dim))
label_info = []
index = -1
with torch.no_grad():
    for data in tqdm(dataloader):
        images, labels = data[:2]
        images = images.to(device)
        embeddings = model(images)[0]
        embeddings = embeddings.detach().cpu().numpy()
        labels = label_encoder.inverse_transform(labels)
        label_info += labels.tolist()
        for embedding in embeddings:
            index += 1
            embedding_info[index] = embedding
embedding_info = pd.DataFrame(embedding_info, index=None)


In [ ]:
plt.subplots(figsize=(10, 10))
tsne = TSNE(n_components=2, init="random", learning_rate="auto")
tsne_info = tsne.fit_transform(embedding_info)
color = list(CSS4_COLORS.values())
color = [color[int(i)] for i in label_info]
plt.scatter(tsne_info[:, 0], tsne_info[:, 1], color=color)
plt.show()
plt.close()


In [ ]:
label_info = label_info.tolist()